<a href="https://colab.research.google.com/github/yangyi02/droid/blob/main/pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DROID Multi-View 3D Tracking Pipeline

This notebook is a **thin orchestration layer** — all algorithm code lives in the GitHub repo.

**Four stages** controlled by global flags (Stages 1–3 can compute or load from GCS):

| Stage | Compute | Load from GCS | Output |
|---|---|---|---|
| 1. Depth | `compute_depth.py` | `gs://dm-tapnet/tmp/droid/depth/` | Stereo depth + gripper refinement |
| 2. Extrinsics | `compute_extrinsics.py` | `gs://dm-tapnet/tmp/droid/extrinsics/` | Camera-robot alignment |
| 3. Tracks | `compute_tracks.py` | `gs://dm-tapnet/tmp/droid/tracks/` | Static BG + Robot 3D tracks |
| 4. Metrics | `compute_metrics.py` | — | Quality metrics + visualization |

---
## 0. Environment Setup

In [ ]:
import importlib.util
import os

IN_COLAB = (importlib.util.find_spec("google") is not None
            and importlib.util.find_spec("google.colab") is not None)

if IN_COLAB:
    REPO_DIR = "/content/droid"
    if not os.path.exists(REPO_DIR):
        !git clone --recursive https://github.com/yangyi02/droid.git {REPO_DIR}
    else:
        !cd {REPO_DIR} && git pull && git submodule update --init --recursive
    %cd {REPO_DIR}
else:
    REPO_DIR = os.getcwd()
    while not os.path.exists(os.path.join(REPO_DIR, "core", "io.py")):
        parent = os.path.dirname(REPO_DIR)
        if parent == REPO_DIR:
            REPO_DIR = os.getcwd()
            break
        REPO_DIR = parent

if not os.path.exists(os.path.join(REPO_DIR, "core", "io.py")):
    raise SystemExit(
        f"{REPO_DIR} is not the droid repo root -- open this notebook from "
        "the checkout, or set REPO_DIR by hand.")

CACHE_DIR = os.path.join(REPO_DIR, "data", "cache")

print(f"{'Colab' if IN_COLAB else 'Local'}: {REPO_DIR}")
print(f"cache: {CACHE_DIR}")

In [ ]:

COMPUTE_DEPTH      = False
COMPUTE_EXTRINSICS = False
COMPUTE_TRACKS     = False

print(f"COMPUTE_DEPTH      = {COMPUTE_DEPTH}")
print(f"COMPUTE_EXTRINSICS = {COMPUTE_EXTRINSICS}")
print(f"COMPUTE_TRACKS     = {COMPUTE_TRACKS}")

In [ ]:
if not IN_COLAB:
    print("[SKIP] local checkout — dependencies come from setup.sh / the venv")
else:

    !pip install -q mediapy pyrender
    !pip install --no-binary pybullet --no-build-isolation --no-cache-dir pybullet
    import pybullet
    if not pybullet.isNumpyEnabled():
        print("  [WARN] pybullet built without NumPy support -- "
              "getCameraImage will be ~15x slower than it should be")

    if COMPUTE_DEPTH:
        print("\n[Depth] Installing ZED SDK + model weights...")
        import shutil
        if not shutil.which("ZED_Explorer"):
            !apt-get update -qq
            !apt-get install -y zstd
            sdk_installer = "ZED_SDK_Linux_Ubuntu22.run"
            !wget -q -O {sdk_installer} https://download.stereolabs.com/zedsdk/5.2/cu12/ubuntu22
            !chmod +x {sdk_installer}
            !./{sdk_installer} silent runtime_only skip_tools
            !find /usr/local/zed/ -name "pyzed*.whl" -exec pip install {} \;
            !pip install -q "numpy==2.0.2"
            print("  ZED SDK installed")
        else:
            print("  [SKIP] ZED SDK already installed")
        !pip install -q open3d
        import os
        if not os.path.exists("third_party/s2m2/src"):
            print("  Initializing s2m2 submodule...")
            !git submodule update --init --recursive
        !pip install -q git+https://github.com/facebookresearch/segment-anything.git
        os.makedirs("third_party/s2m2/weights", exist_ok=True)
        os.makedirs("third_party/sam_weights", exist_ok=True)
        !wget -nc -q -O third_party/s2m2/weights/CH384NTR3.pth \
            "https://huggingface.co/minimok/s2m2/resolve/main/CH384NTR3.pth"
        !wget -nc -q -O third_party/sam_weights/sam_vit_h_4b8939.pth \
            "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth"
        print("  Depth dependencies ready")

    if COMPUTE_EXTRINSICS:
        print("\n[Extrinsics] No extra model weights needed (dataset extrinsics + differentiable rendering).")

    if COMPUTE_TRACKS:
        print("\n[Tracks] No extra model weights needed (static prior + URDF FK).")

    print("\nDependency install complete.")

In [ ]:
import importlib
import os
import random
import subprocess
import sys

import cv2
import matplotlib.pyplot as plt
import mediapy as media
import numpy as np
import torch

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

if COMPUTE_DEPTH:
    s2m2_path = os.path.join(REPO_DIR, "third_party/s2m2/src")
    if s2m2_path not in sys.path:
        sys.path.insert(0, s2m2_path)

os.environ['PYOPENGL_PLATFORM'] = 'egl'
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")


In [ ]:
if IN_COLAB:
    result = subprocess.run(["git", "pull", "--ff-only"], cwd=REPO_DIR,
                            capture_output=True, text=True)
    print(result.stdout.strip() or result.stderr.strip())

import core.depth
import core.geometry
import core.io
import core.physics
import core.tracking
import core.visualization
import compute_depth
import compute_extrinsics
import compute_metrics
import compute_tracks

for mod in [core.depth, core.geometry, core.io, core.physics, core.tracking,
            core.visualization, compute_depth, compute_extrinsics,
            compute_metrics, compute_tracks]:
    importlib.reload(mod)

print("All modules reloaded. Re-run cells below to test changes.")

In [ ]:
serials_db, id_to_path, keep_ranges, extrinsics_db, _ = core.io.load_metadata()

with open(os.path.join(REPO_DIR, "episodes_success.txt")) as f:
    valid_ids = sorted(line.strip() for line in f if line.strip())
print(f"{len(valid_ids)} successful episodes to pick from")

In [ ]:

episode_id = random.choice(valid_ids)


print(f"Episode: {episode_id}")

In [ ]:
scene_constants = compute_depth.init_episode(
    episode_id,
    os.path.join(core.io.INPUT_ROOT, "robotics", "droid_raw", "1.0.1"),
    id_to_path, serials_db, keep_ranges)
print(f"scene_constants initialized: {list(scene_constants['camera'].keys())}")

---
## 1. Stage 1: Depth

Stereo depth via S2M2 + SAM gripper refinement.

In [ ]:
if COMPUTE_DEPTH:
    if 's2m2_model' not in dir():
        s2m2_model, sam_predictor, run_stereo_matching = compute_depth.init_all_models()

    scene_constants = compute_depth.extract_svo_video(scene_constants)
    scene_constants = compute_depth.parse_robot_kinematics(scene_constants)
    scene_constants = compute_depth.align_temporal_streams(scene_constants)
    scene_constants = core.depth.compute_stereo_depth(
        scene_constants, s2m2_model, run_stereo_matching, device)

    wrist_serial = scene_constants["meta"].get("wrist_serial")
    if wrist_serial and wrist_serial in scene_constants["camera"]:
        wrist_data = scene_constants["camera"][wrist_serial]
        if "raw_depth" in wrist_data:
            wrist_data["original_raw_depth"] = wrist_data["raw_depth"].copy()
    scene_constants = core.depth.build_universal_gripper_mask(
        scene_constants, sam_predictor)
    scene_constants = core.depth.distill_empirical_gripper_depth(scene_constants)
    scene_constants = core.depth.inject_gripper_depth(scene_constants)

    compute_depth.export_depth(scene_constants)
    print("Stage 1 (Depth) COMPUTE complete")

else:
    GCS_DEPTH = "gs://dm-tapnet/tmp/droid/depth"
    depth_root = os.path.join(CACHE_DIR, "depth")
    ep_cache = os.path.join(depth_root, episode_id)
    os.makedirs(ep_cache, exist_ok=True)

    os.system(f"gsutil cp '{GCS_DEPTH}/{episode_id}/robot.npz' '{ep_cache}/' > /dev/null 2>&1")
    wrist_serial = str(np.load(f"{ep_cache}/robot.npz", allow_pickle=True)["wrist_serial"])

    for cam in scene_constants['camera']:
        cam_dir = os.path.join(ep_cache, cam)
        os.makedirs(cam_dir, exist_ok=True)
        files = ["video_left.mp4", "video_right.mp4", "raw_depth.npz", "calibration.npz"]
        if cam == wrist_serial:
            files += ["original_raw_depth.npz", "gripper_mask.npz", "gripper_depth.npz"]
        srcs = " ".join(f"'{GCS_DEPTH}/{episode_id}/{cam}/{f}'" for f in files)
        os.system(f"gsutil -m cp {srcs} '{cam_dir}/' > /dev/null 2>&1")
        print(f"  fetched {cam}")

    scene_constants = core.io.load_depth_data(episode_id, depth_root,
                                              load_video="full", inspection=True)
    print("Stage 1 LOADED from GCS")

In [ ]:
core.visualization.inspect_dict_structure(scene_constants)

frames = core.visualization.render_multicam_disparity_video(scene_constants, max_frames=30)
media.show_video(frames, fps=10, title="Depth [left | right | disparity] per camera")

In [ ]:
core.visualization.render_gripper_refinement_inspection(scene_constants, frame_idx=0)

wrist_serial = scene_constants['meta'].get('wrist_serial')
if wrist_serial and wrist_serial in scene_constants['camera']:
    cam_data = scene_constants['camera'][wrist_serial]
    emp_depth = cam_data.get('empirical_gripper_depth')
    if emp_depth is not None and np.any(emp_depth > 0):
        core.visualization.render_distilled_gripper_3d(
            median_depth=emp_depth,
            K_mat=cam_data['K_mat'],
            rgb_img=cam_data['video_rgb'][0]
        )

---
## 2. Stage 2: Extrinsics

Dataset extrinsics → differentiable robot alignment → global joint optimization.

In [ ]:
if COMPUTE_EXTRINSICS:
    pb_renderer = core.physics.PyBulletRenderer(gpu=True)
    print(f"PyBulletRenderer EGL: {pb_renderer.gpu}")

    scene_state = compute_extrinsics.init_camera_states(scene_constants, extrinsics_db)

    scene_state = compute_extrinsics.per_camera_alignment(
        scene_constants, pb_renderer, scene_state, device)
    
    pb_renderer = core.physics.PyBulletRenderer(gpu=True)
    scene_state = compute_extrinsics.global_joint_alignment(
        scene_constants, scene_state, pb_renderer, device)

    compute_extrinsics.export_extrinsics(scene_constants, scene_state)
    print("Stage 2 (Extrinsics) COMPUTE complete")

else:
    GCS_EXT = "gs://dm-tapnet/tmp/droid/extrinsics"
    ext_root = os.path.join(CACHE_DIR, "extrinsics")
    for cam in scene_constants['camera']:
        cam_dir = os.path.join(ext_root, episode_id, cam)
        os.makedirs(cam_dir, exist_ok=True)
        os.system(f"gsutil cp '{GCS_EXT}/{episode_id}/{cam}/extrinsics.json' "
                  f"'{cam_dir}/extrinsics.json' > /dev/null 2>&1")

    scene_state = core.io.load_extrinsics(scene_constants, ext_root)

    pb_renderer = core.physics.PyBulletRenderer(gpu=True)

    metrics = compute_metrics.evaluate_extrinsics(
        scene_constants, scene_state, device, pb_renderer=pb_renderer)
    compute_metrics.print_metrics(metrics, "Loaded Extrinsics")
    print("Extrinsics LOADED from GCS")

In [ ]:
axes_frames = core.visualization.render_cross_camera_axes(
    scene_constants, scene_state, max_frames=30)
if axes_frames:
    media.show_video(axes_frames, fps=10, title="Camera Axes Overlay")

In [ ]:
pb_renderer = core.physics.PyBulletRenderer(gpu=True)

seg_frames = core.visualization.render_segmentation_video(
    scene_constants, scene_state, pb_renderer, max_frames=30)
if seg_frames:
    media.show_video(seg_frames, fps=10, title="Robot Mask Overlay")

In [ ]:
core.visualization.render_fused_point_cloud(
    scene_constants, scene_state, frame_idx=0, height=600, width=1000)

In [ ]:
orbit_frames = core.visualization.render_cinematic_4d_orbit(
    scene_constants, scene_state, max_frames=30)
media.show_video(orbit_frames, fps=10, title="4D Orbit")

---
## 3. Stage 3: Tracking

**Static Background + Robot Tracks** — no tracker model dependency.

**Dual-Track Architecture:**
- **Track A (Static Background)**: Multi-view depth consensus → fixed world 3D → project to 2D per-view using extrinsics (static prior)
- **Track B (Robot)**: URDF forward kinematics → per-link binding → cross-view projection

**Output format** (`tracks_3d.npz`):
- `traj_3d`: (T, N, 3) — 3D world-frame trajectories
- `vis_global`: (T, N) — global visibility mask

**Per-camera** (`{cam_id}/tracks_2d.npz`):
- `traj_2d`: (T, N, 2) — 2D pixel coordinates
- `vis_2d`: (T, N) — per-camera visibility

In [ ]:
if COMPUTE_TRACKS:
    NUM_STATIC_POINTS = 300
    MAX_ROBOT_PTS_PER_CAM = 100

    camera_ids = list(scene_constants['camera'].keys())
    T_frames = len(scene_constants['camera'][camera_ids[0]]['video_rgb'])

    static_pts_3d, static_rgb = compute_tracks.find_static_candidates(
        scene_constants, scene_state, pb_renderer,
        num_points=NUM_STATIC_POINTS)

    if len(static_pts_3d) > 0:
        static_per_cam_tracks, static_per_cam_vis = compute_tracks.project_static_tracks(
            static_pts_3d, scene_constants, scene_state, pb_renderer)
    else:
        static_per_cam_tracks = {
            cam: np.zeros((T_frames, 0, 2), dtype=np.float32)
            for cam in camera_ids}
        static_per_cam_vis = {
            cam: np.zeros((T_frames, 0), dtype=bool)
            for cam in camera_ids}

    n_static = len(static_pts_3d)

    robot_traj_3d, robot_per_cam_tracks, robot_per_cam_vis, n_robot = \
        compute_tracks.compute_robot_tracks(
            scene_constants, scene_state, pb_renderer,
            max_robot_pts_per_cam=MAX_ROBOT_PTS_PER_CAM)

    (final_traj_3d, final_vis_global, final_per_cam_tracks,
     final_per_cam_vis, n_static, n_robot) = compute_tracks.merge_tracks(
        static_pts_3d, static_per_cam_tracks, static_per_cam_vis,
        robot_traj_3d, robot_per_cam_tracks, robot_per_cam_vis,
        camera_ids, T_frames)

    compute_tracks.export_tracks(scene_constants, scene_state,
                                 final_traj_3d, final_vis_global,
                                 final_per_cam_tracks, final_per_cam_vis,
                                 n_static, n_robot)

    print(f"\nStage 3 COMPUTE complete: {n_static} static + {n_robot} robot = {final_traj_3d.shape[1]} points")

else:
    GCS_TRACKS = "gs://dm-tapnet/tmp/droid/tracks"
    tracks_root = os.path.join(CACHE_DIR, "tracks")
    ep_cache = os.path.join(tracks_root, episode_id)
    os.makedirs(ep_cache, exist_ok=True)

    for fname in ["tracks_3d.npz", "track_metadata.npz"]:
        os.system(f"gsutil cp '{GCS_TRACKS}/{episode_id}/{fname}' "
                  f"'{ep_cache}/' > /dev/null 2>&1")
    for cam_id in scene_constants["camera"]:
        cam_cache = os.path.join(ep_cache, cam_id)
        os.makedirs(cam_cache, exist_ok=True)
        srcs = " ".join(f"'{GCS_TRACKS}/{episode_id}/{cam_id}/{f}'"
                        for f in ["tracks_2d.npz", "intrinsics.npy", "extrinsics_w2c.npy"])
        os.system(f"gsutil -m cp {srcs} '{cam_cache}/' > /dev/null 2>&1")

    tracks = compute_metrics.load_track_data(episode_id, tracks_root)
    if tracks is None:
        raise RuntimeError(f"No tracks on GCS for {episode_id} -- "
                           "set COMPUTE_TRACKS = True to make them.")

    final_traj_3d        = tracks["traj_3d"]
    final_vis_global     = tracks["vis_global"]
    final_per_cam_tracks = tracks["per_cam_tracks"]
    final_per_cam_vis    = tracks["per_cam_vis"]
    n_static, n_robot    = tracks["n_static"], tracks["n_robot"]

    T, N, _ = final_traj_3d.shape
    print(f"Stage 3 LOADED from GCS — {len(final_per_cam_tracks)} cameras, "
          f"{T} frames, {n_static} static + {n_robot} robot = {N} points")

In [ ]:
camera_ids = list(scene_constants['camera'].keys())

ref_cam = camera_ids[min(1, len(camera_ids) - 1)]
if n_static > 0:
    y_static = final_per_cam_tracks[ref_cam][0, :n_static, 1]
    norm_s = plt.Normalize(y_static.min(), y_static.max())
    static_colors = (plt.cm.gist_rainbow(norm_s(y_static))[:, :3] * 255).astype(np.uint8)
else:
    static_colors = np.zeros((0, 3), dtype=np.uint8)

robot_colors = np.full((n_robot, 3), [255, 50, 50], dtype=np.uint8) if n_robot > 0 else np.zeros((0, 3), dtype=np.uint8)
combined_colors = np.concatenate([static_colors, robot_colors], axis=0)

all_frames_static = []
for cam_id in camera_ids:
    cam_data = scene_constants['camera'][cam_id]
    tracks = final_per_cam_tracks[cam_id][:, :n_static, :]
    vis = final_per_cam_vis[cam_id][:, :n_static]
    frames = core.visualization.render_2d_tracking_video(
        cam_data['video_rgb'], tracks, vis,
        global_colors=static_colors,
        tgt_size=(256, 456), linewidth=1, max_frames=60)
    for f in frames:
        cv2.putText(f, f"Cam [{cam_id[:8]}]", (10, 25),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 3)
        cv2.putText(f, f"Cam [{cam_id[:8]}]", (10, 25),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
    all_frames_static.append(np.array(frames))

if all_frames_static:
    combined = np.concatenate(all_frames_static, axis=2)
    media.show_video(combined, fps=10,
                     title=f"Static Background Tracks ({n_static} points) — All Cameras")

if n_robot > 0:
    all_frames_robot = []
    for cam_id in camera_ids:
        cam_data = scene_constants['camera'][cam_id]
        tracks = final_per_cam_tracks[cam_id][:, n_static:, :]
        vis = final_per_cam_vis[cam_id][:, n_static:]
        frames = core.visualization.render_2d_tracking_video(
            cam_data['video_rgb'], tracks, vis,
            global_colors=robot_colors,
            tgt_size=(256, 456), linewidth=1, max_frames=60)
        for f in frames:
            cv2.putText(f, f"Cam [{cam_id[:8]}]", (10, 25),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 3)
            cv2.putText(f, f"Cam [{cam_id[:8]}]", (10, 25),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
        all_frames_robot.append(np.array(frames))

    if all_frames_robot:
        combined_robot = np.concatenate(all_frames_robot, axis=2)
        media.show_video(combined_robot, fps=10,
                         title=f"Robot Tracks ({n_robot} points) — All Cameras")

all_frames_both = []
for cam_id in camera_ids:
    cam_data = scene_constants['camera'][cam_id]
    frames = core.visualization.render_2d_tracking_video(
        cam_data['video_rgb'],
        final_per_cam_tracks[cam_id],
        final_per_cam_vis[cam_id],
        global_colors=combined_colors,
        tgt_size=(256, 456), linewidth=1, max_frames=60)
    for f in frames:
        cv2.putText(f, f"Cam [{cam_id[:8]}]", (10, 25),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 3)
        cv2.putText(f, f"Cam [{cam_id[:8]}]", (10, 25),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 1)
    all_frames_both.append(np.array(frames))

if all_frames_both:
    combined_both = np.concatenate(all_frames_both, axis=2)
    media.show_video(combined_both, fps=10,
                     title=f"All Tracks ({n_static} static (static) + {n_robot} robot (robot))")

In [ ]:
print(f"Using 'final_traj_3d', shape={final_traj_3d.shape}")

orbit_frames = core.visualization.render_4d_orbit_with_tracks(
    scene_constants, scene_state,
    tracks_3d=final_traj_3d,
    track_history=5,
    track_sphere_radius=0.006,
    frustum_depth=0.12,
    max_render_points=300000,
    max_render_tracks=500,
    width=960, height=540,
    orbit_center=(0.4, 0.0, 0.0),
    orbit_radius=1.2,
    camera_height=0.5,
    max_frames=60,
)
media.show_video(orbit_frames, fps=10,
                 title="4D Orbit — Point Cloud + Tracks + Cameras")

---
## 4. Stage 4: Quality Metrics

Evaluate pipeline output quality using `compute_metrics` — the same functions called by `compute_metrics.py` (GCP batch runner) and `tapvidmv/select_episodes.py` (episode selection).

| Cell | What it shows | Function |
|---|---|---|
| 4a | All metrics summary (scene, motion, extrinsics, depth, tracks) | `evaluate_episode()` |
| 4b | Extrinsics detailed visualization (robot depth + chamfer curves) | `evaluate_extrinsics()` |
| 4c | Depth consistency histogram (per-camera, static vs robot) | `compute_depth_residual_per_camera()` |

In [ ]:
pb_renderer = core.physics.PyBulletRenderer(gpu=True)

all_metrics = compute_metrics.evaluate_episode(
    scene_constants, scene_state, device,
    final_traj_3d=final_traj_3d,
    final_per_cam_vis=final_per_cam_vis,
    n_static=n_static,
    n_robot=n_robot,
    compute_extrinsics_metrics=True,
    pb_renderer=pb_renderer,
)

print(f"Episode: {all_metrics['episode_id']}")
print(f"{'=' * 60}")

sections = {
    "Scene": ["site", "robot_id", "n_cameras", "image_resolution", "n_frames"],
    "Motion": ["ee_travel_m", "joint_range_mean_rad", "joint_range_max_rad", "gripper_range"],
    "Extrinsics": ["chamfer_total", "robot_loss_cam1", "robot_loss_cam2", "robot_loss_wrist"],
    "Depth Coverage": [k for k in sorted(all_metrics) if k.startswith("depth_coverage") or k.startswith("depth_median") or k.startswith("depth_range")],
    "Track Depth Consistency": [k for k in sorted(all_metrics) if k.startswith("depth_residual")],
    "Track Visibility": [k for k in sorted(all_metrics) if k.startswith("vis_")],
}

for section_name, keys in sections.items():
    print(f"\n  {section_name}:")
    for k in keys:
        v = all_metrics.get(k, "—")
        if isinstance(v, float):
            print(f"    {k:45s} = {v:.4f}")
        else:
            print(f"    {k:45s} = {v}")

In [ ]:
pb_renderer = core.physics.PyBulletRenderer(gpu=True)

wrist_cam = scene_constants['meta']['wrist_serial']
ext_cams = [c for c in scene_constants['camera'].keys() if c != wrist_cam]
n_frames = len(scene_constants['robot']['joint_positions'])
T_ee_all = scene_constants['robot']['T_ee_base_all']
VIS_FRAME = n_frames // 2

all_cam_ids = list(ext_cams) + [wrist_cam]
cam_labels = ['cam1 (ext)', 'cam2 (ext)', 'wrist']

joints_vis = scene_constants['robot']['joint_positions'][VIS_FRAME]
gripper_vis = scene_constants['robot']['gripper_positions'][VIS_FRAME]
pb_renderer.update_robot_pose(joints_vis, gripper_state=gripper_vis)

fig, axes = plt.subplots(3, len(all_cam_ids), figsize=(6*len(all_cam_ids), 14))
for col, (cam_id, label) in enumerate(zip(all_cam_ids, cam_labels)):
    K_np = scene_constants['camera'][cam_id]['K_mat']
    ext_t = scene_state[cam_id]['extrinsics'][VIS_FRAME]
    d_obs = scene_constants['camera'][cam_id]['raw_depth'][VIS_FRAME].astype(np.float32)
    h_img, w_img = d_obs.shape
    d_render = pb_renderer.render_depth(ext_t, K_np, w_img, h_img)
    robot_mask = d_render > 0.01

    im0 = axes[0, col].imshow(np.where(robot_mask, d_render, np.nan), cmap='viridis', vmin=0.1, vmax=1.2)
    axes[0, col].set_title(f'{label}\nRendered Robot Depth', fontsize=11); axes[0, col].axis('off')
    plt.colorbar(im0, ax=axes[0, col], fraction=0.046)

    im1 = axes[1, col].imshow(np.where(robot_mask, d_obs, np.nan), cmap='viridis', vmin=0.1, vmax=1.2)
    axes[1, col].set_title(f'{label}\nObserved Sensor Depth\n(robot region)', fontsize=11); axes[1, col].axis('off')
    plt.colorbar(im1, ax=axes[1, col], fraction=0.046)

    diff = np.abs(d_render - d_obs)
    valid = robot_mask & (d_obs > 0.01) & (d_obs < 1.5)
    diff_masked = np.where(valid, diff, np.nan)
    im2 = axes[2, col].imshow(diff_masked, cmap='hot', vmin=0, vmax=0.1)
    mean_err = np.nanmean(diff_masked) if np.any(valid) else 0
    axes[2, col].set_title(f'{label}\n|Rendered - Observed|\nmean={mean_err:.4f}m', fontsize=11); axes[2, col].axis('off')
    plt.colorbar(im2, ax=axes[2, col], fraction=0.046)
plt.suptitle(f'Robot Depth Loss Inputs (Frame {VIS_FRAME})', fontsize=14, y=1.02)
plt.tight_layout(); plt.show()

print("  Robot segmentation overlay:")
core.visualization.render_multiview_mask_inspection(
    scene_constants, scene_state, pb_renderer, frame_idx=VIS_FRAME)

fig, axes_curve = plt.subplots(1, 3, figsize=(18, 4))
for ax, cam_id, label in zip(axes_curve, all_cam_ids, cam_labels):
    K_np = scene_constants['camera'][cam_id]['K_mat']
    per_frame_err = []
    for t in range(n_frames):
        joints = scene_constants['robot']['joint_positions'][t]
        gripper = scene_constants['robot']['gripper_positions'][t]
        pb_renderer.update_robot_pose(joints, gripper_state=gripper)
        ext_t = scene_state[cam_id]['extrinsics'][t]
        d_obs = scene_constants['camera'][cam_id]['raw_depth'][t].astype(np.float32)
        h_img, w_img = d_obs.shape
        d_render = pb_renderer.render_depth(ext_t, K_np, w_img, h_img)
        valid = (d_render > 0.01) & (d_render < 1.5) & (d_obs > 0.01) & (d_obs < 1.5)
        per_frame_err.append(np.abs(d_render[valid] - d_obs[valid]).mean() if valid.any() else np.nan)
    per_frame_err = np.array(per_frame_err)
    ax.plot(per_frame_err, linewidth=0.8)
    ax.axhline(y=np.nanmean(per_frame_err), color='r', linestyle='--',
               label=f'mean={np.nanmean(per_frame_err):.4f}m')
    ax.set_xlabel('Frame'); ax.set_ylabel('Mean |Δ depth| (m)')
    ax.set_title(f'{label} [{cam_id[:8]}]'); ax.legend(fontsize=8)
plt.suptitle('Robot Depth Loss per Frame', fontsize=14)
plt.tight_layout(); plt.show()

cam1, cam2 = ext_cams[0], ext_cams[1]
T1 = torch.tensor(scene_state[cam1]['base_extrinsic'], dtype=torch.float32, device=device)
T2 = torch.tensor(scene_state[cam2]['base_extrinsic'], dtype=torch.float32, device=device)
Tw = torch.tensor(scene_state[wrist_cam]['base_extrinsic'], dtype=torch.float32, device=device)

chamfer_12, chamfer_1w, chamfer_2w = [], [], []
for t in range(n_frames):
    pc1 = compute_extrinsics.get_cam_points_local_t(
        t, scene_constants['camera'][cam1], device)
    pc2 = compute_extrinsics.get_cam_points_local_t(
        t, scene_constants['camera'][cam2], device)
    pcw = compute_extrinsics.get_cam_points_local_t(
        t, scene_constants['camera'][wrist_cam], device)
    if pc1 is None or pc2 is None or pcw is None:
        chamfer_12.append(np.nan); chamfer_1w.append(np.nan); chamfer_2w.append(np.nan)
        continue
    Tee_t = torch.tensor(T_ee_all[t], dtype=torch.float32, device=device)
    w1 = (T1 @ pc1)[:3, :].T.unsqueeze(0)
    w2 = (T2 @ pc2)[:3, :].T.unsqueeze(0)
    ww = ((Tee_t @ Tw) @ pcw)[:3, :].T.unsqueeze(0)
    l12, _ = compute_extrinsics.batched_chamfer_distance(w1, w2, device)
    l1w, _ = compute_extrinsics.batched_chamfer_distance(w1, ww, device)
    l2w, _ = compute_extrinsics.batched_chamfer_distance(w2, ww, device)
    chamfer_12.append(l12.item()); chamfer_1w.append(l1w.item()); chamfer_2w.append(l2w.item())

fig, ax = plt.subplots(1, 1, figsize=(12, 4))
ax.plot(chamfer_12, label=f'cam1↔cam2 (mean={np.nanmean(chamfer_12):.4f})', linewidth=0.8)
ax.plot(chamfer_1w, label=f'cam1↔wrist (mean={np.nanmean(chamfer_1w):.4f})', linewidth=0.8)
ax.plot(chamfer_2w, label=f'cam2↔wrist (mean={np.nanmean(chamfer_2w):.4f})', linewidth=0.8)
ax.set_xlabel('Frame'); ax.set_ylabel('Chamfer Distance (m)')
ax.set_title('Chamfer Distance per Frame (lower = better alignment)')
ax.legend(); plt.tight_layout(); plt.show()

metrics = compute_metrics.evaluate_extrinsics(
    scene_constants, scene_state, device, pb_renderer=pb_renderer)
compute_metrics.print_metrics(metrics, f"Extrinsics Summary")

fig, ax = plt.subplots(1, 1, figsize=(10, 5))
depth_names = ['chamfer_total', 'robot_loss_cam1', 'robot_loss_cam2', 'robot_loss_wrist']
depth_vals = [metrics.get(k, 0) for k in depth_names]
depth_labels = ['Chamfer\ntotal', 'Robot\ncam1', 'Robot\ncam2', 'Robot\nwrist']
bars = ax.bar(depth_labels, depth_vals, color=['#2196F3', '#FF9800', '#FF9800', '#FF9800'])
for bar, val in zip(bars, depth_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
            f'{val:.4f}', ha='center', va='bottom', fontsize=9)
ax.set_ylabel('Error (meters)'); ax.set_title('3D Extrinsics Metrics')
plt.suptitle(f'Extrinsics Quality Summary — {episode_id}', fontsize=14)
plt.tight_layout(); plt.show()

In [ ]:
camera_ids = list(scene_constants['camera'].keys())

errors = compute_metrics.compute_depth_residual_per_camera(
    scene_constants, scene_state,
    final_traj_3d, final_per_cam_vis, n_static, n_robot)

plt.rcParams.update({'font.sans-serif': 'DejaVu Sans', 'axes.edgecolor': '#cccccc', 'axes.linewidth': 0.8})
fig, axes = plt.subplots(1, len(camera_ids), figsize=(4.8 * len(camera_ids), 3.8), sharey=True)
if len(camera_ids) == 1:
    axes = [axes]

palette = {'static': '#2da44e', 'robot': '#cf222e', 'all': '#0969da'}

for ax, cam_id in zip(axes, camera_ids):
    s_err = errors[cam_id]['static']
    r_err = errors[cam_id]['robot']
    a_err = errors[cam_id]['all']

    if len(s_err):
        med_s = np.median(s_err)
        ax.hist(s_err, bins=40, range=(0, 40), alpha=0.4, color=palette['static'], label=f'Static (Med: {med_s:.1f} mm)')
        ax.axvline(med_s, color=palette['static'], linestyle='--', linewidth=1.2)
    if len(r_err):
        med_r = np.median(r_err)
        ax.hist(r_err, bins=40, range=(0, 40), alpha=0.4, color=palette['robot'], label=f'Robot (Med: {med_r:.1f} mm)')
        ax.axvline(med_r, color=palette['robot'], linestyle='--', linewidth=1.2)
    if len(a_err):
        med_a = np.median(a_err)
        ax.axvline(med_a, color=palette['all'], linestyle='-', linewidth=1.6, label=f'Overall (Med: {med_a:.1f} mm)')

    ax.set_title(f'Camera [{cam_id[:8]}]', fontsize=11, pad=10, fontweight='bold')
    ax.set_xlabel('Depth Residual Error (mm)', fontsize=9)
    ax.grid(True, linestyle=':', alpha=0.5)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    ax.legend(frameon=True, facecolor='white', framealpha=0.95, fontsize=8)

axes[0].set_ylabel('Observation Count', fontsize=9)
plt.suptitle(f'Depth Consistency (Static={n_static}, Robot={n_robot}, Total={final_traj_3d.shape[1]})', fontsize=12, y=1.03)
plt.tight_layout(); plt.show()